# 🦷 SmileGuard — Dental Detection Training Pipeline

**6-Class YOLOv8m Dental Disease Detection** · Max 1,000 images per class

| Step | Description |
|------|-------------|
| 1 | **Setup** — Imports, paths, class definitions |
| 2 | **Merge** — Combine 5 Roboflow datasets with class remapping (≤1K/class) |
| 3 | **Validate** — Label sanity checks, bbox verification |
| 4 | **Analyze** — Class distribution, dataset statistics |
| 5 | **Train** — YOLOv8m, 200 epochs, AdamW |
| 6 | **Monitor** — Plateau detection, generalization gap |
| 7 | **Evaluate** — Per-class metrics on test set |
| 8 | **Inference** — DentalDetector class + visualization |
| 9 | **Export** — Flask API for integration |

### Target Classes
| ID | Class | Description |
|----|-------|-------------|
| 0 | `caries` | Dental decay / rot |
| 1 | `cavity` | Holes in teeth |
| 2 | `crack` | Tooth fractures / cracks |
| 3 | `tooth` | Healthy tooth baseline |
| 4 | `gingivitis` | Gum disease / inflammation |
| 5 | `calculus` | Tartar buildup |

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 1 — Setup & Configuration
# ═══════════════════════════════════════════════════════════════

import os, shutil, yaml, random, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter
from ultralytics import YOLO

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

# ── PATHS ──
BASE_DIR = Path(r"C:\Users\PerezKylerLee(Studen\SmileGuard-Train")
DATASET_DIR = Path(r"C:\Users\PerezKylerLee(Studen\Downloads\Dataset")
OUTPUT_DIR = BASE_DIR / "smileguard_merged"
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# ── 5 SOURCE DATASETS (already downloaded, YOLO format) ──
SOURCE_DATASETS = {
    "Dental_Caries":     DATASET_DIR / "Dental caries.v2i.yolov8",
    "Dental_Datasets":   DATASET_DIR / "Dental Datasets.v3i.yolov8",
    "Dental_Detecting":  DATASET_DIR / "dental detectin2.v1i.yolov8",
    "Dental_Detection":  DATASET_DIR / "Dental Detection.v1i.yolov8",
    "Dental_Disease":    DATASET_DIR / "Dental Disease.v3i.yolov8",
}

# ── TARGET 6 CLASSES ──
TARGET_CLASSES = ["caries", "cavity", "crack", "tooth", "gingivitis", "calculus"]
TARGET_ID = {name: i for i, name in enumerate(TARGET_CLASSES)}
MAX_IMAGES_PER_CLASS = 1000  # hard cap per class

# ── CLASS REMAPPING (every source class name → target class) ──
CLASS_MAP = {
    # caries (decay → caries, but cavity stays separate)
    "Caries": "caries", "caries": "caries", "cariess": "caries",
    "decay": "caries",
    # cavity (separate from caries now)
    "Cavity": "cavity", "cavity": "cavity",
    # crack
    "Crack": "crack", "crack": "crack",
    # tooth
    "Tooth": "tooth", "tooth": "tooth", "teeth": "tooth", "Teeth": "tooth",
    # gingivitis
    "gingivitis": "gingivitis", "Gingivitis": "gingivitis",
    # calculus
    "calculus": "calculus", "Calculus": "calculus", "CALCULUS": "calculus",
    # SKIP these (not in our 6 target classes)
    # "hypodontia" → skip, "ulcer" → skip, "Fillings" → skip,
    # "Impacted Tooth" → skip, "Implant" → skip, "infected-teeth" → skip,
    # "abscess" → skip, "tooth-discoloration" → skip, "Tooth Discoloration" → skip
}

# ── Verify datasets exist ──
print("=" * 70)
print("  SmileGuard — Setup Complete")
print("=" * 70)
print(f"\n  Base dir:    {BASE_DIR}")
print(f"  Dataset dir: {DATASET_DIR}")
print(f"  Output dir:  {OUTPUT_DIR}")
print(f"  Classes:     {TARGET_CLASSES}")
print(f"  Max/class:   {MAX_IMAGES_PER_CLASS:,}")

print(f"\n  Source datasets:")
all_found = True
for name, path in SOURCE_DATASETS.items():
    exists = path.exists()
    status = "✅" if exists else "❌ NOT FOUND"
    print(f"    {name:20s} → {status}")
    if not exists:
        all_found = False

if all_found:
    print(f"\n  ✅ All 5 datasets found. Ready to merge.")
else:
    print(f"\n  ❌ Missing datasets! Check paths above.")

## Step 2 — Merge Datasets

Merges all 5 Roboflow datasets into `smileguard_merged/` with:
- Class remapping to unified 5-class schema
- **Skips** dummy full-image boxes (`w≥0.99 AND h≥0.99`) — the root cause of the 7% mAP disaster
- **Skips** out-of-range coordinates
- **Skips** unmapped classes (Crack, Fillings, Implant, etc.)
- Handles filename collisions
- Maps `test` + `valid` → `val`

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 2 — Merge All Datasets (local, no downloads needed)
# ═══════════════════════════════════════════════════════════════

def polygon_to_bbox(coords):
    """Convert polygon vertices [x1,y1,x2,y2,...] → YOLO xywh."""
    xs = coords[0::2]  # every even index
    ys = coords[1::2]  # every odd index
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    x_c = (x_min + x_max) / 2.0
    y_c = (y_min + y_max) / 2.0
    w = x_max - x_min
    h = y_max - y_min
    return x_c, y_c, w, h


def validate_and_remap_label(label_path, src_classes, unmapped_log, format_stats=None):
    """Read YOLO label, remap class IDs, skip bad boxes.
    Handles BOTH standard YOLO (5 parts) and polygon/segmentation (>5 parts)."""
    remapped = []
    with open(label_path, 'r') as f:
        lines = f.read().strip().splitlines()

    for line in lines:
        if not line.strip():
            continue
        parts = line.strip().split()
        if len(parts) < 5:
            continue

        try:
            cls_id = int(parts[0])
            coords = [float(p) for p in parts[1:]]
        except (ValueError, IndexError):
            continue

        # Standard YOLO bbox: cls_id x_c y_c w h (5 parts total)
        if len(parts) == 5:
            x_c, y_c, w, h = coords
            if format_stats is not None:
                format_stats["bbox"] += 1
        # Polygon/segmentation: cls_id x1 y1 x2 y2 ... (>5 parts, even number of coords)
        elif len(coords) >= 4 and len(coords) % 2 == 0:
            x_c, y_c, w, h = polygon_to_bbox(coords)
            if format_stats is not None:
                format_stats["polygon"] += 1
        else:
            continue  # malformed

        # Skip out-of-range coordinates
        if not all(0.0 <= c <= 1.0 for c in [x_c, y_c, w, h]):
            continue

        # Skip near-zero area boxes
        if w < 0.001 or h < 0.001:
            continue

        # CRITICAL: Skip full-image dummy boxes (THE bug that killed training)
        if w >= 0.99 and h >= 0.99:
            continue

        # Remap class name
        if cls_id >= len(src_classes):
            continue
        src_name = src_classes[cls_id]
        target_name = CLASS_MAP.get(src_name)

        if target_name is None:
            unmapped_log.add(src_name)
            continue

        target_id = TARGET_ID[target_name]
        remapped.append(f"{target_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}")

    return remapped if remapped else None


# ── Clean output directory ──
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
for split in ["train", "val"]:
    (OUTPUT_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("  MERGING 5 DATASETS → smileguard_merged/")
print(f"  Per-class cap: {MAX_IMAGES_PER_CLASS:,} images")
print("=" * 70)

stats = Counter()
unmapped_classes = set()
existing_names = {"train": set(), "val": set()}
format_stats = Counter()  # track bbox vs polygon label formats
class_image_counts = Counter()  # per-class image cap tracker

for ds_name, ds_path in SOURCE_DATASETS.items():
    # Read source data.yaml
    yaml_path = ds_path / "data.yaml"
    if not yaml_path.exists():
        print(f"\n  ❌ {ds_name}: No data.yaml found, skipping")
        continue

    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    src_classes = cfg.get("names", [])
    print(f"\n  📂 {ds_name}")
    print(f"     Source classes: {src_classes}")

    # Find splits
    for split_name in ["train", "valid", "val", "test"]:
        img_dir = ds_path / split_name / "images"
        lbl_dir = ds_path / split_name / "labels"
        if not img_dir.exists() or not lbl_dir.exists():
            continue

        # Map everything to train or val
        out_split = "val" if split_name in ("valid", "val", "test") else "train"
        out_img = OUTPUT_DIR / out_split / "images"
        out_lbl = OUTPUT_DIR / out_split / "labels"

        split_added = 0
        for lbl_path in sorted(lbl_dir.glob("*.txt")):
            remapped = validate_and_remap_label(str(lbl_path), src_classes, unmapped_classes, format_stats)
            if remapped is None:
                stats["skipped_labels"] += 1
                continue

            # ── Per-class cap: skip image if ANY of its classes already hit 1K ──
            img_classes = set()
            for ann in remapped:
                cid = int(ann.split()[0])
                img_classes.add(TARGET_CLASSES[cid])
            if any(class_image_counts[c] >= MAX_IMAGES_PER_CLASS for c in img_classes):
                stats["capped"] += 1
                continue
            # Increment counters for all classes present in this image
            for c in img_classes:
                class_image_counts[c] += 1

            # Find matching image
            stem = lbl_path.stem
            img_path = None
            for ext in VALID_EXTS:
                candidate = img_dir / (stem + ext)
                if candidate.exists():
                    img_path = candidate
                    break
            if img_path is None:
                stats["no_image"] += 1
                continue

            # Handle filename collisions
            img_ext = img_path.suffix
            new_name = f"{stem}{img_ext}"
            if new_name in existing_names[out_split]:
                counter = 1
                while f"{stem}_{counter}{img_ext}" in existing_names[out_split]:
                    counter += 1
                new_name = f"{stem}_{counter}{img_ext}"
            existing_names[out_split].add(new_name)
            new_stem = Path(new_name).stem

            # Copy image + write remapped label
            shutil.copy2(str(img_path), str(out_img / new_name))
            with open(out_lbl / f"{new_stem}.txt", 'w') as f:
                f.write("\n".join(remapped) + "\n")

            split_added += 1
            stats[f"{out_split}_added"] += 1
            stats["total_instances"] += len(remapped)

        print(f"     {split_name:5s} → {out_split}: {split_added:,} images")

# ── Write data.yaml ──
data_yaml = {
    "path": str(OUTPUT_DIR.resolve()),
    "train": "train/images",
    "val": "val/images",
    "nc": len(TARGET_CLASSES),
    "names": TARGET_CLASSES,
}
with open(OUTPUT_DIR / "data.yaml", 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

# ── Summary ──
print(f"\n{'=' * 70}")
print(f"  MERGE COMPLETE")
print(f"{'=' * 70}")
print(f"  Train images:          {stats['train_added']:,}")
print(f"  Val images:            {stats['val_added']:,}")
print(f"  Total bbox instances:  {stats['total_instances']:,}")
print(f"  Skipped (bad labels):  {stats['skipped_labels']:,}")
print(f"  Skipped (no image):    {stats['no_image']:,}")
print(f"  Capped (≥1K/class):    {stats['capped']:,}")
print(f"  Label formats:         {format_stats['bbox']:,} bbox | {format_stats['polygon']:,} polygon→bbox")
print(f"\n  Per-class image counts (capped at {MAX_IMAGES_PER_CLASS}):")
for cls_name in TARGET_CLASSES:
    print(f"    {cls_name:<20s}: {class_image_counts[cls_name]:>5,}")
print(f"\n  Output: {OUTPUT_DIR}")
print(f"  data.yaml written ✓")

if unmapped_classes:
    print(f"\n  ⚠ UNMAPPED CLASSES (skipped — not in our 6 targets):")
    for c in sorted(unmapped_classes):
        print(f"    - '{c}'")
else:
    print(f"\n  ✅ All classes mapped successfully")

## Step 3 — Validate Labels & BBox Sanity

**Critical checkpoint** — This is where we catch the corruption that killed the first training run.

A healthy dataset should have:
- Avg box width/height: `0.05–0.35` (small-to-medium dental features)
- Zero boxes at `(0.5, 0.5, 1.0, 1.0)` (dummy full-image boxes)
- All coordinates in `[0, 1]`
- Matching image/label counts

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 3 — Label Validation & BBox Sanity Check
# ═══════════════════════════════════════════════════════════════

import statistics

print("=" * 70)
print("  LABEL VALIDATION & BBOX SANITY CHECK")
print("=" * 70)

all_healthy = True

for split in ["train", "val"]:
    lbl_dir = OUTPUT_DIR / split / "labels"
    img_dir = OUTPUT_DIR / split / "images"

    if not lbl_dir.exists():
        continue

    images = {f.stem for f in img_dir.iterdir() if f.suffix.lower() in VALID_EXTS}
    labels = {f.stem for f in lbl_dir.glob("*.txt")}

    n_imgs = len(images)
    n_lbls = len(labels)
    missing_labels = images - labels
    orphan_labels = labels - images

    widths, heights, centers_x, centers_y = [], [], [], []
    dummy_count = 0
    bad_coords = 0
    class_counts = Counter()
    total_boxes = 0

    for lbl_file in sorted(lbl_dir.glob("*.txt")):
        for line in lbl_file.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            try:
                cls_id = int(parts[0])
                xc, yc, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            except ValueError:
                continue

            total_boxes += 1
            class_counts[cls_id] += 1
            widths.append(w)
            heights.append(h)
            centers_x.append(xc)
            centers_y.append(yc)

            if w >= 0.99 and h >= 0.99:
                dummy_count += 1
            if not all(0 <= v <= 1 for v in [xc, yc, w, h]):
                bad_coords += 1

    print(f"\n  📊 {split.upper()} SPLIT:")
    print(f"     Images: {n_imgs:,}  |  Labels: {n_lbls:,}  |  Match: {'✅' if n_imgs == n_lbls else '❌ MISMATCH'}")

    if missing_labels:
        print(f"     ⚠ {len(missing_labels)} images without labels")
    if orphan_labels:
        print(f"     ⚠ {len(orphan_labels)} orphan labels (no image)")

    if widths:
        avg_w = statistics.mean(widths)
        avg_h = statistics.mean(heights)
        print(f"\n     Bounding Box Stats:")
        print(f"       Total boxes:     {total_boxes:,}")
        print(f"       Avg box width:   {avg_w:.4f}  {'✅' if 0.01 < avg_w < 0.5 else '🚨 SUSPICIOUS'}")
        print(f"       Avg box height:  {avg_h:.4f}  {'✅' if 0.01 < avg_h < 0.5 else '🚨 SUSPICIOUS'}")
        print(f"       Max box width:   {max(widths):.4f}")
        print(f"       Max box height:  {max(heights):.4f}")
        print(f"       Dummy (1.0x1.0): {dummy_count}  {'✅ ZERO' if dummy_count == 0 else '🚨🚨🚨 CORRUPTED!'}")
        print(f"       Bad coords:      {bad_coords}  {'✅' if bad_coords == 0 else '⚠'}")

        if dummy_count > 0:
            all_healthy = False
        if avg_w > 0.8 or avg_h > 0.8:
            all_healthy = False

    print(f"\n     Per-class instances:")
    for cid in sorted(class_counts.keys()):
        name = TARGET_CLASSES[cid] if cid < len(TARGET_CLASSES) else f"UNKNOWN_{cid}"
        flag = " ⚠ OUT OF RANGE" if cid >= len(TARGET_CLASSES) else ""
        print(f"       {cid}={name:20s}: {class_counts[cid]:>6,}{flag}")
        if cid >= len(TARGET_CLASSES):
            all_healthy = False

# ── VISUAL: BBox scatter plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Center scatter
axes[0].scatter(centers_x[:3000], centers_y[:3000], alpha=0.15, s=3, c='steelblue')
axes[0].set_xlim(0, 1)
axes[0].set_ylim(1, 0)
axes[0].set_xlabel("x_center")
axes[0].set_ylabel("y_center")
axes[0].set_title("BBox Centers (should be scattered)")
axes[0].set_aspect('equal')

# Width-Height scatter
axes[1].scatter(widths[:3000], heights[:3000], alpha=0.15, s=3, c='coral')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].set_xlabel("width")
axes[1].set_ylabel("height")
axes[1].set_title("BBox Sizes (should NOT cluster at 1.0, 1.0)")
axes[1].set_aspect('equal')

plt.suptitle("SmileGuard Label Quality — BBox Distribution", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR / "label_sanity_check.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'=' * 70}")
if all_healthy:
    print("  ✅✅✅ ALL LABELS HEALTHY — SAFE TO TRAIN ✅✅✅")
else:
    print("  🚨🚨🚨 LABEL ISSUES DETECTED — DO NOT TRAIN YET 🚨🚨🚨")
print(f"{'=' * 70}")

## Step 4 — Class Distribution Analysis

Visual breakdown of the merged dataset.
Check that no class is severely under/over-represented before training.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 4 — Class Distribution Analysis
# ═══════════════════════════════════════════════════════════════

with open(OUTPUT_DIR / "data.yaml") as f:
    data_cfg = yaml.safe_load(f)
NC = data_cfg["nc"]
NAMES = data_cfg["names"]

print("=" * 80)
print("  CLASS DISTRIBUTION ANALYSIS")
print("=" * 80)

split_counts = {}
for split in ["train", "val"]:
    counts = Counter()
    lbl_dir = OUTPUT_DIR / split / "labels"
    if lbl_dir.exists():
        for lbl in lbl_dir.glob("*.txt"):
            for line in lbl.read_text().strip().splitlines():
                parts = line.strip().split()
                if parts:
                    try:
                        counts[int(parts[0])] += 1
                    except ValueError:
                        pass
    split_counts[split] = counts
    n_imgs = len(list((OUTPUT_DIR / split / "images").glob("*")))
    print(f"\n  {split.upper()}: {n_imgs:,} images, {sum(counts.values()):,} annotations")

# Combined
total = Counter()
for c in split_counts.values():
    total += c
total_annots = sum(total.values())

print(f"\n  {'ID':<4} {'Class':<22} {'Train':>8} {'Val':>8} {'Total':>8} {'%':>7}")
print(f"  {'-'*60}")
for cid in range(NC):
    tr = split_counts["train"].get(cid, 0)
    va = split_counts["val"].get(cid, 0)
    tot = total.get(cid, 0)
    pct = (tot / total_annots * 100) if total_annots else 0
    print(f"  {cid:<4} {NAMES[cid]:<22} {tr:>8,} {va:>8,} {tot:>8,} {pct:>6.1f}%")

# Imbalance
nonzero = [total.get(cid, 0) for cid in range(NC) if total.get(cid, 0) > 0]
if len(nonzero) >= 2:
    ratio = max(nonzero) / min(nonzero)
    print(f"\n  Imbalance ratio: {ratio:.1f}x (max/min)")
    if ratio > 5:
        print(f"  ⚠ High imbalance — consider augmentation for minority classes")
    else:
        print(f"  ✅ Acceptable balance")

# ── Bar chart ──
fig, ax = plt.subplots(figsize=(10, 5))
counts_list = [total.get(cid, 0) for cid in range(NC)]
colors = ['#e74c3c', '#c0392b', '#f39c12', '#2ecc71', '#e67e22', '#3498db']
bars = ax.barh(NAMES, counts_list, color=colors, edgecolor='gray')
for bar, cnt in zip(bars, counts_list):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2, f'{cnt:,}',
            va='center', fontsize=10)
ax.set_xlabel("Number of Annotations")
ax.set_title("SmileGuard — Class Distribution (Merged Dataset)")
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(BASE_DIR / "class_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"\n  📊 Chart saved to class_distribution.png")

## Step 5 — Train YOLOv8m

**Configuration from Handoff:**
- Backbone: `yolov8m.pt` (52 MB, medium — accuracy/speed sweet spot)
- 200 epochs, patience 50, AdamW optimizer
- Heavy augmentation: mosaic 0.8, copy_paste 0.1, mixup 0.05
- Dropout 0.1, warmup 5 epochs, close_mosaic at epoch 190

### 🚨 Red Flags to Watch For
| Signal | Meaning |
|--------|---------|
| mAP50 < 0.10 after epoch 30 | Label/path problem |
| train/box_loss ↓ but val/box_loss ↑ | Overfitting |
| precision OR recall stuck ≈ 0 | Class mapping broken |
| Only ~300 instances/epoch | Labels not loading |

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 5 — Train YOLOv8m (per Handoff spec)
# ═══════════════════════════════════════════════════════════════

DATA_YAML = str((OUTPUT_DIR / "data.yaml").resolve())
MODEL_PATH = str(BASE_DIR / "yolov8m.pt")

assert Path(DATA_YAML).exists(), f"data.yaml not found at {DATA_YAML}"
assert Path(MODEL_PATH).exists(), f"Model weights not found at {MODEL_PATH}"

print(f"  Data config : {DATA_YAML}")
print(f"  Model       : {MODEL_PATH}")
print(f"  Device      : cuda:0 (auto-fallback to cpu)")

# ── Load model ──
model = YOLO(MODEL_PATH)

# ── Train ──
results = model.train(
    data=DATA_YAML,
    epochs=200,
    imgsz=640,
    batch=16,
    patience=50,
    dropout=0.1,
    optimizer="AdamW",
    device=0,
    amp=True,
    seed=0,

    # Augmentation (aggressive, per handoff)
    mosaic=0.8,
    copy_paste=0.1,
    mixup=0.05,
    degrees=10.0,
    close_mosaic=10,

    # Scheduling
    warmup_epochs=5,
    save_period=5,

    # Project structure
    project=str(BASE_DIR / "runs" / "detect" / "Dental_Detection"),
    name="YOLOv8_Training2",
    exist_ok=True,

    # Logging
    verbose=True,
    plots=True,
)

print("\n" + "=" * 70)
print("  TRAINING COMPLETE")
print("=" * 70)
TRAIN_DIR = Path(results.save_dir)
print(f"  Results: {TRAIN_DIR}")
print(f"  Best weights: {TRAIN_DIR / 'weights' / 'best.pt'}")
print(f"  Last weights: {TRAIN_DIR / 'weights' / 'last.pt'}")

## Step 6 — Training Analysis & Plateau Detection

Post-training (or mid-training) diagnostic. Reads `results.csv` from the run directory.

**What to check:**
- Loss curves (train vs val) — divergence = overfitting
- mAP trajectory — should climb steadily
- Generalization gap — val loss shouldn't be >2x train loss

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 6 — Training Analysis & Plateau Detection
# ═══════════════════════════════════════════════════════════════

# Auto-detect latest run directory
TRAIN_DIR = TRAIN_DIR if 'TRAIN_DIR' in dir() else sorted(
    (BASE_DIR / "runs" / "detect" / "Dental_Detection").glob("YOLOv8_Training*"),
    key=lambda p: p.stat().st_mtime
)[-1]

results_csv = TRAIN_DIR / "results.csv"
assert results_csv.exists(), f"results.csv not found at {results_csv}"

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()
print(f"  Loaded {len(df)} epochs from {results_csv.name}")

# ── Key metrics ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1) Box Loss: train vs val
ax = axes[0, 0]
ax.plot(df["epoch"], df["train/box_loss"], label="Train", color="#2ecc71", linewidth=1.5)
ax.plot(df["epoch"], df["val/box_loss"], label="Val", color="#e74c3c", linewidth=1.5)
ax.set_title("Box Loss")
ax.set_xlabel("Epoch")
ax.legend()
ax.grid(alpha=0.3)

# 2) Cls Loss: train vs val
ax = axes[0, 1]
ax.plot(df["epoch"], df["train/cls_loss"], label="Train", color="#2ecc71", linewidth=1.5)
ax.plot(df["epoch"], df["val/cls_loss"], label="Val", color="#e74c3c", linewidth=1.5)
ax.set_title("Classification Loss")
ax.set_xlabel("Epoch")
ax.legend()
ax.grid(alpha=0.3)

# 3) mAP
ax = axes[1, 0]
if "metrics/mAP50(B)" in df.columns:
    ax.plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50", color="#3498db", linewidth=2)
if "metrics/mAP50-95(B)" in df.columns:
    ax.plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95", color="#9b59b6", linewidth=1.5, linestyle="--")
ax.set_title("mAP Trajectory")
ax.set_xlabel("Epoch")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)

# 4) Precision & Recall
ax = axes[1, 1]
if "metrics/precision(B)" in df.columns:
    ax.plot(df["epoch"], df["metrics/precision(B)"], label="Precision", color="#e67e22", linewidth=1.5)
if "metrics/recall(B)" in df.columns:
    ax.plot(df["epoch"], df["metrics/recall(B)"], label="Recall", color="#1abc9c", linewidth=1.5)
ax.set_title("Precision & Recall")
ax.set_xlabel("Epoch")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle("SmileGuard Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(BASE_DIR / "training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Plateau detection ──
print("\n" + "=" * 70)
print("  TRAINING HEALTH REPORT")
print("=" * 70)

best_epoch = df["metrics/mAP50(B)"].idxmax() if "metrics/mAP50(B)" in df.columns else -1
best_mAP = df["metrics/mAP50(B)"].max() if "metrics/mAP50(B)" in df.columns else 0
last_epoch = int(df["epoch"].iloc[-1])

print(f"  Best mAP50:    {best_mAP:.4f} (epoch {best_epoch})")
print(f"  Final epoch:   {last_epoch}")

# Generalization gap (last 10 epochs average)
tail = df.tail(10)
train_loss_avg = tail["train/box_loss"].mean()
val_loss_avg = tail["val/box_loss"].mean()
gap = val_loss_avg / train_loss_avg if train_loss_avg > 0 else 0
print(f"  Gen. gap (last 10): train_loss={train_loss_avg:.3f}, val_loss={val_loss_avg:.3f}, ratio={gap:.2f}x")

if gap > 2.0:
    print("  ⚠ Overfitting detected — val loss >> train loss")
elif gap > 1.5:
    print("  ⚡ Mild overfitting — consider more augmentation or early stopping")
else:
    print("  ✅ Healthy generalization")

# Check for early stagnation
if last_epoch >= 30 and best_mAP < 0.10:
    print("  🚨 CRITICAL: mAP50 never exceeded 10% — likely a data/label problem!")
elif best_mAP >= 0.60:
    print(f"  🎯 Strong model: mAP50 = {best_mAP:.1%}")
elif best_mAP >= 0.30:
    print(f"  📈 Decent progress: mAP50 = {best_mAP:.1%} — room to improve")

print(f"\n  📊 Curves saved to training_curves.png")

## Step 7 — Model Evaluation

Load best weights and evaluate on validation set.
Per-class breakdown of precision, recall, mAP50, mAP50-95.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 7 — Model Evaluation
# ═══════════════════════════════════════════════════════════════

BEST_WEIGHTS = TRAIN_DIR / "weights" / "best.pt"
assert BEST_WEIGHTS.exists(), f"best.pt not found at {BEST_WEIGHTS}"

best_model = YOLO(str(BEST_WEIGHTS))
print(f"  Loaded best weights: {BEST_WEIGHTS}")

# ── Validate on merged val set ──
val_results = best_model.val(
    data=DATA_YAML,
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    verbose=True,
)

# ── Per-class metrics table ──
print("\n" + "=" * 80)
print("  PER-CLASS EVALUATION RESULTS")
print("=" * 80)

header = f"  {'Class':<22} {'P':>8} {'R':>8} {'mAP50':>8} {'mAP50-95':>10}"
print(header)
print(f"  {'-' * 60}")

class_names = TARGET_CLASSES
box = val_results.box

# Per-class
for i, name in enumerate(class_names):
    p = box.p[i] if i < len(box.p) else 0
    r = box.r[i] if i < len(box.r) else 0
    ap50 = box.ap50[i] if i < len(box.ap50) else 0
    ap = box.ap[i] if i < len(box.ap) else 0
    print(f"  {name:<22} {p:>8.3f} {r:>8.3f} {ap50:>8.3f} {ap:>10.3f}")

# Overall
print(f"  {'-' * 60}")
print(f"  {'ALL':<22} {box.mp:>8.3f} {box.mr:>8.3f} {box.map50:>8.3f} {box.map:>10.3f}")

print(f"\n  Confusion matrix & PR curves saved in: {val_results.save_dir}")

# ── Pass/fail summary ──
print("\n  VERDICT:")
if box.map50 >= 0.75:
    print(f"  🏆 EXCELLENT — mAP50 = {box.map50:.1%}")
elif box.map50 >= 0.50:
    print(f"  ✅ GOOD — mAP50 = {box.map50:.1%} — production viable")
elif box.map50 >= 0.30:
    print(f"  📈 IMPROVING — mAP50 = {box.map50:.1%} — needs more data/epochs")
else:
    print(f"  ⚠ LOW — mAP50 = {box.map50:.1%} — investigate data quality")

## Step 8 — DentalDetector Wrapper & Inference

Production-ready detection class that wraps the YOLO model.
Test on sample validation images with annotated visualizations.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 8 — DentalDetector Class + Visual Inference
# ═══════════════════════════════════════════════════════════════

class DentalDetector:
    """SmileGuard dental disease detector — wraps YOLOv8 best.pt."""

    CLASS_NAMES = ["caries", "cavity", "crack", "tooth", "gingivitis", "calculus"]
    COLORS = {
        "caries": (231, 76, 60),
        "cavity": (192, 57, 43),
        "crack": (243, 156, 18),
        "tooth": (46, 204, 113),
        "gingivitis": (230, 126, 34),
        "calculus": (52, 152, 219),
    }

    def __init__(self, weights_path, conf=0.25, iou=0.45, device=0):
        self.model = YOLO(str(weights_path))
        self.conf = conf
        self.iou = iou
        self.device = device
        print(f"  DentalDetector loaded: {weights_path}")
        print(f"  Classes: {self.CLASS_NAMES}")
        print(f"  Confidence: {conf}, IoU: {iou}")

    def predict(self, image_path, save=False, save_dir=None):
        """Run detection on a single image."""
        results = self.model.predict(
            source=str(image_path),
            conf=self.conf,
            iou=self.iou,
            device=self.device,
            verbose=False,
            save=save,
            project=str(save_dir) if save_dir else None,
        )
        return results[0]

    def annotate(self, image_path):
        """Run detection and return annotated image + detections list."""
        result = self.predict(image_path)
        img = cv2.imread(str(image_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        detections = []
        for box in result.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            name = self.CLASS_NAMES[cls_id] if cls_id < len(self.CLASS_NAMES) else f"class_{cls_id}"
            color = self.COLORS.get(name, (200, 200, 200))

            detections.append({
                "class": name,
                "confidence": conf,
                "bbox": [x1, y1, x2, y2],
            })

            # Draw box
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            label = f"{name} {conf:.0%}"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(img, (x1, y1 - th - 6), (x1 + tw, y1), color, -1)
            cv2.putText(img, label, (x1, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                        (255, 255, 255), 1, cv2.LINE_AA)

        return img, detections

    def batch_summary(self, image_paths):
        """Run on multiple images and return summary stats."""
        all_dets = []
        for p in image_paths:
            _, dets = self.annotate(p)
            all_dets.extend(dets)
        class_counts = Counter(d["class"] for d in all_dets)
        avg_conf = sum(d["confidence"] for d in all_dets) / len(all_dets) if all_dets else 0
        return {"total_detections": len(all_dets), "per_class": dict(class_counts), "avg_confidence": avg_conf}


# ── Instantiate ──
detector = DentalDetector(BEST_WEIGHTS, conf=0.25)

# ── Inference on 8 random val images ──
val_imgs = list((OUTPUT_DIR / "val" / "images").glob("*"))
sample = random.sample(val_imgs, min(8, len(val_imgs)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, img_path in zip(axes.flat, sample):
    annotated, dets = detector.annotate(img_path)
    ax.imshow(annotated)
    ax.set_title(f"{len(dets)} detection(s)", fontsize=10)
    ax.axis("off")

# Hide empty axes
for ax in axes.flat[len(sample):]:
    ax.axis("off")

plt.suptitle("SmileGuard — Sample Detections (Val Set)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(BASE_DIR / "sample_detections.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Batch summary ──
summary = detector.batch_summary(sample)
print(f"\n  Sample batch summary ({len(sample)} images):")
print(f"  Total detections: {summary['total_detections']}")
print(f"  Avg confidence:   {summary['avg_confidence']:.1%}")
for cls, cnt in sorted(summary["per_class"].items()):
    print(f"    {cls}: {cnt}")

## Step 9 — Export & Flask API

Export best model to ONNX/TorchScript for deployment.
Minimal Flask API for serving predictions.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 9 — Export Best Model
# ═══════════════════════════════════════════════════════════════

export_model = YOLO(str(BEST_WEIGHTS))

# ONNX export (universal deployment)
onnx_path = export_model.export(format="onnx", imgsz=640, simplify=True)
print(f"  ✅ ONNX exported: {onnx_path}")

# TorchScript export (PyTorch mobile)
ts_path = export_model.export(format="torchscript", imgsz=640)
print(f"  ✅ TorchScript exported: {ts_path}")

# Copy best.pt to a clean location
deploy_dir = BASE_DIR / "deploy"
deploy_dir.mkdir(exist_ok=True)
shutil.copy2(str(BEST_WEIGHTS), str(deploy_dir / "smileguard_best.pt"))
print(f"  📦 Deploy-ready weights: {deploy_dir / 'smileguard_best.pt'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Flask API (run in a separate terminal, NOT inside the notebook)
# Save this to app.py with: File > Save As
# ═══════════════════════════════════════════════════════════════

FLASK_APP_CODE = '''
"""SmileGuard Dental Detection API — Flask"""
import io, json
from pathlib import Path
from flask import Flask, request, jsonify
from PIL import Image
from ultralytics import YOLO

app = Flask(__name__)
MODEL_PATH = "deploy/smileguard_best.pt"
model = YOLO(MODEL_PATH)
CLASS_NAMES = ["caries", "cavity", "crack", "tooth", "gingivitis", "calculus"]

@app.route("/predict", methods=["POST"])
def predict():
    if "image" not in request.files:
        return jsonify({"error": "No image uploaded"}), 400

    file = request.files["image"]
    img = Image.open(io.BytesIO(file.read())).convert("RGB")
    results = model.predict(source=img, conf=0.25, iou=0.45, verbose=False)

    detections = []
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        detections.append({
            "class": CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else f"class_{cls_id}",
            "confidence": round(float(box.conf[0]), 3),
            "bbox": [round(c, 1) for c in box.xyxy[0].tolist()],
        })

    return jsonify({
        "detections": detections,
        "count": len(detections),
        "classes_found": list(set(d["class"] for d in detections)),
    })

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": MODEL_PATH, "classes": CLASS_NAMES})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

# Write Flask app to file
app_path = BASE_DIR / "app.py"
app_path.write_text(FLASK_APP_CODE.strip())
print(f"  ✅ Flask API written to: {app_path}")
print(f"\n  To run:  python app.py")
print(f"  Test:    curl -X POST -F 'image=@test.jpg' http://localhost:5000/predict")
print(f"\n  ── SmileGuard Pipeline Complete ──")